# Notebook 9 — is $\alpha_H$ the right parameter?

Before attempting a lower bound it is worth asking whether the upper bound is tight where the
known lower bounds are tight. This notebook measures $m$, $\kappa$, $\alpha_H$ and $\#K_3$ on
the instances the literature uses, and compares three quantities:

| bound | meaning |
|---|---|
| $m\,\alpha_H/\#K_3$ | ours, with a perfect predictor |
| $m\,\kappa/\#K_3$ | Bera and Seshadhri, prediction-free and constant-pass |
| $\min\{m^{3/2}/\#K_3,\ m/\sqrt{\#K_3}\}$ | Bera and Chakrabarti, prediction-free lower bound |

The comparison with Bera and Seshadhri is the one that matters for a separation claim, since
their algorithm is also constant-pass: an instance witnesses a separation only if predictions
beat the degeneracy ordering too. That is exactly what $\alpha_H/\kappa$ measures.

**A prior warning about the model.** If an algorithm could use a perfect value oracle freely,
the problem would be trivial: maintain $\sum_e \hat w(e)$ in one pass and $O(\log n)$ space and
return it divided by three. What keeps the model of the earlier paper non-degenerate is the
$+1$ smoothing together with invariance to a global scale, so a distortion-$1$ predictor
determines only $c\,(3\#H+m)$ for an unknown $c$. Any lower bound must therefore restrict the
oracle: a threshold oracle, a distortion bounded away from one, or a bounded number of
queries. Everything below concerns the upper bound only.

## 0. Library

In [ ]:
"""Core graph primitives.

Degeneracy by peeling, exact oracle-width (pseudoarboricity) by binary search over
max-flow feasibility, and the structural bracket

    ceil(kappa_H / 2)  <=  alpha_H  <=  kappa_H  <=  kappa(G)

of Lemma 1 in the paper.
"""
import time
from collections import defaultdict

import numpy as np
from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import maximum_flow

__all__ = ['bits_to_idx', 'popcount', 'relabel', 'canon', 'build_adj',
           'degeneracy', 'degeneracy_arrays', 'orientation_feasible', 'oracle_width',
           'bracket']


# ----------------------------------------------------------------- bit helpers

def bits_to_idx(x, nbytes):
    """Indices of the set bits of a Python int."""
    if x == 0:
        return np.empty(0, dtype=np.int32)
    b = np.frombuffer(x.to_bytes(nbytes, 'little'), dtype=np.uint8)
    return np.flatnonzero(np.unpackbits(b, bitorder='little')).astype(np.int32)


def popcount(x):
    try:
        return x.bit_count()
    except AttributeError:              # Python < 3.10
        return bin(x).count('1')


# ------------------------------------------------------------- edge-list helpers

def relabel(edges):
    """Map arbitrary hashable node labels to consecutive integers."""
    ids, out = {}, []
    for u, v in edges:
        for x in (u, v):
            if x not in ids:
                ids[x] = len(ids)
        out.append((ids[u], ids[v]))
    return out, ids


def canon(edges):
    """De-duplicate, drop self-loops, relabel to ints, return sorted (u, v) with u < v."""
    edges, _ = relabel(edges)
    s = set()
    for u, v in edges:
        if u == v:
            continue
        s.add((u, v) if u < v else (v, u))
    return sorted(s)


def build_adj(edges):
    adj = defaultdict(set)
    for u, v in edges:
        if u == v:
            continue
        adj[u].add(v)
        adj[v].add(u)
    return adj


# ------------------------------------------------------------------- degeneracy

def degeneracy(adj):
    """Exact degeneracy of an adjacency dict, by peeling. Returns (kappa, core numbers)."""
    if not adj:
        return 0, {}
    deg = {v: len(adj[v]) for v in adj}
    maxdeg = max(deg.values())
    buckets = [set() for _ in range(maxdeg + 1)]
    for v, d in deg.items():
        buckets[d].add(v)
    core, k, removed = {}, 0, set()
    i = 0
    for _ in range(len(deg)):
        while i <= maxdeg and not buckets[i]:
            i += 1
        if i > maxdeg:
            break
        v = buckets[i].pop()
        k = max(k, i)
        core[v] = k
        removed.add(v)
        for w in adj[v]:
            if w in removed:
                continue
            d = deg[w]
            buckets[d].discard(w)
            deg[w] = d - 1
            buckets[d - 1].add(w)
            if d - 1 < i:
                i = d - 1
    return k, core


def degeneracy_arrays(eu, ev):
    """Degeneracy of the graph given by parallel edge arrays."""
    if len(eu) == 0:
        return 0, 0
    nodes = np.unique(np.concatenate([eu, ev]))
    remap = {int(v): i for i, v in enumerate(nodes)}
    n = len(nodes)
    adj = [[] for _ in range(n)]
    for a, b in zip(eu, ev):
        a, b = remap[int(a)], remap[int(b)]
        adj[a].append(b)
        adj[b].append(a)
    deg = np.array([len(a) for a in adj])
    maxd = int(deg.max())
    buckets = [set() for _ in range(maxd + 1)]
    for v in range(n):
        buckets[deg[v]].add(v)
    removed = np.zeros(n, dtype=bool)
    k, i = 0, 0
    for _ in range(n):
        while i <= maxd and not buckets[i]:
            i += 1
        if i > maxd:
            break
        v = buckets[i].pop()
        k = max(k, i)
        removed[v] = True
        for w in adj[v]:
            if removed[w]:
                continue
            d = int(deg[w])
            buckets[d].discard(w)
            deg[w] = d - 1
            buckets[d - 1].add(w)
            if d - 1 < i:
                i = d - 1
    return k, n


# ------------------------------------------------- oracle-width (pseudoarboricity)

def orientation_feasible(eu, ev, k):
    """Is there an orientation of these edges with maximum out-degree at most k?

    Max-flow feasibility (Hakimi; Frank and Gyarfas):
    s -> edge (cap 1); edge -> each endpoint (cap 1); vertex -> t (cap k).
    Feasible iff the flow saturates all m edges.
    """
    m = len(eu)
    if m == 0:
        return True
    nodes = np.unique(np.concatenate([eu, ev]))
    pos = {int(v): i for i, v in enumerate(nodes)}
    n = len(nodes)
    S, E0, V0 = 0, 1, 1 + m
    T = 1 + m + n
    rows = np.empty(3 * m + n, dtype=np.int64)
    cols = np.empty(3 * m + n, dtype=np.int64)
    data = np.empty(3 * m + n, dtype=np.int32)
    idx = np.arange(m)
    rows[:m] = S
    cols[:m] = E0 + idx
    data[:m] = 1
    rows[m:2 * m] = E0 + idx
    cols[m:2 * m] = V0 + np.array([pos[int(x)] for x in eu])
    data[m:2 * m] = 1
    rows[2 * m:3 * m] = E0 + idx
    cols[2 * m:3 * m] = V0 + np.array([pos[int(x)] for x in ev])
    data[2 * m:3 * m] = 1
    rows[3 * m:] = V0 + np.arange(n)
    cols[3 * m:] = T
    data[3 * m:] = int(k)
    g = csr_matrix((data, (rows, cols)), shape=(T + 1, T + 1))
    return int(maximum_flow(g, S, T).flow_value) == m


def oracle_width(eu, ev, time_budget=None):
    """Exact pseudoarboricity of the edge set; None if the time budget runs out."""
    if len(eu) == 0:
        return 0
    hi, _ = degeneracy_arrays(eu, ev)
    hi = max(hi, 1)
    lo = 1
    t0 = time.time()
    while lo < hi:
        if time_budget is not None and time.time() - t0 > time_budget:
            return None
        mid = (lo + hi) // 2
        if orientation_feasible(eu, ev, mid):
            hi = mid
        else:
            lo = mid + 1
    return lo


def bracket(kappa_copy):
    """The bracket of Lemma 1: (lower bound, upper bound) on alpha_H."""
    return int(np.ceil(kappa_copy / 2)), int(kappa_copy)


"""Synthetic families.

Two of these realise the separation regime (Proposition 4 of the paper) and two are
negative controls that do not.
"""
import numpy as np

__all__ = ['friendship_plus_decoy', 'affine_plane_incidence', 'steiner_plus_butterflies',
           'cliques_plus_cross', 'incidence_edges', 'random_uniform_hypergraph']


def friendship_plus_decoy(s):
    """The separation instance of the PredCount paper: F_s together with K_{d,d}.

    kappa = Theta(sqrt(m)) from the decoy, alpha_{K3} = 2 from the friendship graph.
    """
    d = int(np.ceil(np.sqrt(s)))
    E = []
    for i in range(s):
        E += [('h', ('a', i)), ('h', ('b', i)), (('a', i), ('b', i))]
    for i in range(d):
        for j in range(d):
            E.append((('L', i), ('R', j)))
    return E


def affine_plane_incidence(q):
    """Incidence graph of AG(2, q) for prime q.

    Any two points lie on exactly one line, so the incidence graph is C4-free while its
    degeneracy is Theta(q) = Theta(sqrt(m)): a dense region bearing no copies, from
    geometry rather than construction.
    """
    E = []
    for a in range(q):
        for b in range(q):
            lid = ('L', a, b)
            for x in range(q):
                E.append((('P', x, (a * x + b) % q), lid))
    for c in range(q):
        lid = ('V', c)
        for y in range(q):
            E.append((('P', c, y), lid))
    return E


def steiner_plus_butterflies(q, n_gadgets, r=3):
    """AG(2, q) as a copy-free dense core, plus shallow butterfly gadgets.

    C4-freeness alone is vacuous for counting, so pairs co-occurring in two hyperedges are
    added; they are vertex-disjoint, so the copy-bearing subgraph has constant
    pseudoarboricity.
    """
    E = affine_plane_incidence(q)
    for g in range(n_gadgets):
        a, b = ('G', g, 0), ('G', g, 1)
        for t in range(2):
            h = ('H', g, t)
            E.append((a, h))
            E.append((b, h))
            for j in range(r - 2):
                E.append((('F', g, t, j), h))
    return E


def cliques_plus_cross(n_cliques, clique_size, n_gadgets, seed=0):
    """Disjoint large hyperedges (copy-free cliques) plus shallow cross-triangle gadgets.

    A hyperedge of size r projects to K_r, so kappa >= r - 1, but no triple inside it is
    cross: the dense region bears no copies.  This is the hypergraph analogue of the
    friendship-plus-decoy instance, and the family on which localization grows as
    m^{rho(K3) - 1} = m^{0.5}.
    """
    rng = np.random.default_rng(seed)
    H, nxt, cliques = [], 0, []
    for _ in range(n_cliques):
        c = list(range(nxt, nxt + clique_size))
        nxt += clique_size
        H.append(tuple(c))
        cliques.append(c)
    for _ in range(n_gadgets):
        c = cliques[rng.integers(len(cliques))]
        u, v = rng.choice(c, size=2, replace=False)
        w = nxt
        nxt += 1
        H.append((int(u), w))
        H.append((int(v), w))
    return H


def random_uniform_hypergraph(n, m_h, r, seed=0):
    rng = np.random.default_rng(seed)
    return [tuple(rng.choice(n, size=r, replace=False)) for _ in range(m_h)]


def incidence_edges(hyperedges):
    """Bipartite incidence graph of a hyperedge list."""
    E = []
    for i, h in enumerate(hyperedges):
        for v in set(h):
            E.append((('v', v), ('e', i)))
    return E


"""Known hard instances for streaming triangle counting, measured in our parameters.

The point of this module is to check whether the upper bound of the prediction-augmented
estimator, O~(m * alpha_H / #H), is tight on the instances the literature uses to prove
lower bounds.  If it is, alpha_H is the right parameter and a matching lower bound is worth
attempting; if it is not, the upper bound is loose somewhere and the gap says where.

Three families:

  hubs_graph          Kallaugher and Price's instance, used by Chen et al. for their
                      one-pass lower bound with a heavy-edge oracle.  Every edge lies on at
                      most one triangle, so a heaviness oracle is vacuous on it.
  bera_chakrabarti    the set-disjointness construction behind the constant-pass bound
                      Omega(min{m^{3/2}/T, m/sqrt(T)}).
  friendship_decoy    the separation instance of the PredCount paper (in synthetic.py).

For each we report m, kappa, alpha_H, #K3, and three space bounds: ours, the
degeneracy-based prediction-free bound of Bera and Seshadhri, and the prediction-free
lower bound of Bera and Chakrabarti.  The comparison with Bera and Seshadhri matters,
because that algorithm is also constant-pass: an instance only witnesses a separation if
predictions beat it too, which is exactly what the diagnostic alpha_H / kappa measures.
"""
import numpy as np

__all__ = ['hubs_graph', 'bera_chakrabarti', 'instance_report']


def hubs_graph(r, d):
    """Hub vertex with 2rd incident edges; d disjoint neighbour pairs joined into triangles."""
    E = []
    hub = ('h',)
    for i in range(2 * r * d):
        E.append((hub, ('leaf', i)))
    for j in range(d):                    # pair up 2j and 2j+1 to close a triangle
        E.append((('leaf', 2 * j), ('leaf', 2 * j + 1)))
    return E


def bera_chakrabarti(b, d, n_blocks, x, y):
    """The set-disjointness instance: K_{b,b} plus blocks attached by Alice and Bob.

    A triangle exists for every index i with x_i = y_i = 1 and every (u, v, w) with u in A,
    v in B, w in V_i, so #K3 = b^2 * d * |x AND y|.
    """
    E = []
    A = [('A', i) for i in range(b)]
    B = [('B', i) for i in range(b)]
    for u in A:
        for v in B:
            E.append((u, v))
    for i in range(n_blocks):
        V = [('V', i, t) for t in range(d)]
        if x[i]:
            for u in A:
                for w in V:
                    E.append((u, w))
        if y[i]:
            for v in B:
                for w in V:
                    E.append((v, w))
    return E


def instance_report(edges, label='', exact_alpha=True):
    """Measure the instance and evaluate the three bounds on it."""

    E = canon(edges)
    adj = build_adj(E)
    m = len(E)
    kappa, _ = degeneracy(adj)

    cb, tri = [], 0
    for u in adj:
        for v in adj[u]:
            if u < v:
                c = len(adj[u] & adj[v])
                if c:
                    cb.append((u, v))
                    tri += c
    n_tri = tri // 3
    kappa_copy, _ = degeneracy(build_adj(cb)) if cb else (0, {})
    if cb and exact_alpha:
        eu = np.array([e[0] for e in cb]); ev = np.array([e[1] for e in cb])
        alpha = oracle_width(eu, ev)
    else:
        alpha = 0

    # floor degree: the largest, over triangles, of the least degree of its vertices
    deg = {v: len(adj[v]) for v in adj}
    tau = 0
    for (u, v) in cb:
        for w in adj[u] & adj[v]:
            tau = max(tau, min(deg[u], deg[v], deg[w]))

    ours = (m * alpha / n_tri) if n_tri else np.inf          # O~(m alpha_H / #H)
    bs = (m * kappa / n_tri) if n_tri else np.inf            # Bera and Seshadhri
    bc = (min(m ** 1.5 / n_tri, m / np.sqrt(n_tri))          # Bera and Chakrabarti
          if n_tri else np.inf)
    return dict(
        label=label, m=m, n_triangles=n_tri, kappa=kappa, kappa_copy=kappa_copy,
        alpha=alpha, tau=tau, separable=(tau <= 4 * max(alpha, 1)),
        alpha_over_kappa=alpha / kappa if kappa else np.nan,
        ours=ours, bera_seshadhri=bs, bera_chakrabarti_lb=bc,
        ours_over_lb=ours / bc if bc else np.nan,
        ours_over_bs=ours / bs if bs else np.nan,
    )


In [ ]:
import numpy as np, pandas as pd
pd.set_option('display.width', 260)

rows = []
for r, d in [(1, 64), (1, 256), (4, 64), (8, 64)]:
    rows.append(instance_report(hubs_graph(r, d), f'hubs r={r} d={d}'))
for s in [64, 256, 1024]:
    rows.append(instance_report(friendship_plus_decoy(s), f'friendship+decoy s={s}'))
HARD = pd.DataFrame(rows)
HARD[['label','m','n_triangles','kappa','alpha','tau','separable','alpha_over_kappa',
      'ours','bera_seshadhri','bera_chakrabarti_lb','ours_over_lb','ours_over_bs']].round(3)

## 1. The Bera--Chakrabarti instance, properly parameterised

With $b=N^{s}$ and $d=N^{s-1}$ the construction has $m=\Theta(N^{2s})$ and
$T=\Theta(N^{3s-1})$, and the lower bound evaluates to $\Omega(N)$. Since
$\alpha_H = \Theta(b)$, our bound evaluates to $m\alpha_H/T = \Theta(N)$ as well. The sweep
below checks both claims numerically.

In [ ]:
rows = []
S = 1.5
for N in [6, 9, 12, 16, 20]:
    b = max(2, int(round(N ** S)))
    d = max(1, int(round(N ** (S - 1))))
    ones = max(1, N // 3)
    x = np.zeros(N, int); y = np.zeros(N, int)
    x[:ones] = 1; y[ones - 1:2 * ones - 1] = 1      # exactly one common index
    r = instance_report(bera_chakrabarti(b, d, N, x, y), f'BC N={N} b={b} d={d}')
    r['N'], r['b'] = N, b
    rows.append(r)
    print(f"N={N:>3d} b={b:>3d} d={d}  m={r['m']:>6d} T={r['n_triangles']:>7d} "
          f"kappa={r['kappa']:>3d} alpha={r['alpha']:>3d}  ours={r['ours']:.2f} "
          f"LB={r['bera_chakrabarti_lb']:.2f}")
BC = pd.DataFrame(rows)
BC['ours_over_N'] = BC.ours / BC.N
BC['alpha_over_b'] = BC.alpha / BC.b
BC[['label','m','n_triangles','kappa','alpha','alpha_over_kappa','alpha_over_b',
    'ours','bera_chakrabarti_lb','ours_over_lb','ours_over_N']].round(3)

In [ ]:
print(f"alpha / b            : {BC.alpha_over_b.min():.3f} - {BC.alpha_over_b.max():.3f}"
      f"   (predicted Theta(1))")
print(f"ours / N             : {BC.ours_over_N.min():.3f} - {BC.ours_over_N.max():.3f}"
      f"   (predicted Theta(1))")
print(f"ours / lower bound   : {BC.ours_over_lb.min():.3f} - {BC.ours_over_lb.max():.3f}")
print(f"log-log slope of ours/LB in N: "
      f"{np.polyfit(np.log(BC.N), np.log(BC.ours_over_lb), 1)[0]:+.3f}   (0 means they match)")
print(f"alpha / kappa        : {BC.alpha_over_kappa.min():.3f} - {BC.alpha_over_kappa.max():.3f}")

## 2. Reading the three families together

* **Hubs.** $\alpha_H=\kappa=2$, so our bound coincides with the degeneracy-based bound
  exactly. Predictions buy nothing, and the diagnostic says so in advance. This instance is
  the reason a separation claim has to be made against Bera and Seshadhri rather than against
  the generic $m^{3/2}$ bound: read carelessly, the hubs graph looks like a $\sqrt{d}$
  separation, and it is not one.
* **Bera--Chakrabarti.** Our bound with a perfect predictor is within a small constant of the
  prediction-free lower bound, and scales identically in $N$. On the canonical constant-pass
  hard instance, predictions buy a constant factor and nothing more.
* **Friendship with decoy.** Only here does our bound fall polynomially below both, by exactly
  the factor $\kappa/\alpha_H$.

If the first two rows hold up, the upper bound is tight wherever the known lower bounds are
tight, which is the evidence needed before attempting a matching
$\Omega(m\,\alpha_H/\#H)$ bound, and it also tells us which oracle restriction that bound must
adopt.

In [ ]:
ALL = pd.concat([HARD, BC], ignore_index=True, sort=False)
ALL.to_csv('table16_hard_instances.csv', index=False)
print('wrote table16_hard_instances.csv')
print('\n%%%% ---- Table: our bound on the known hard instances ----')
for _, r in ALL.iterrows():
    print(f"{r.label} & {int(r.m)} & {int(r.n_triangles)} & {int(r.kappa)} & {int(r.alpha)} & "
          f"{r.alpha_over_kappa:.2f} & {r.ours:.1f} & {r.bera_seshadhri:.1f} & "
          f"{r.bera_chakrabarti_lb:.1f} \\\\")